In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px

import func_negocio as func

In [60]:
df_prod = pd.read_csv(r"datos\bd_productividad.csv")

In [61]:
df_prod.groupby(["GOS", "GOR"]).size()

GOS          GOR      
'--          '--          114
Fátima L     Fátima L     266
Hernán P     Daniel R     304
María R      María R      152
Sebastián S  Johanna V    152
             Rodolfo O    285
             Vik E        342
Sofía V      Joe H        209
             José A       209
             Melany A     228
dtype: int64

In [62]:
df_prod1 = df_prod[df_prod["GOS"].isin(['Fátima L', 'Sebastián S', 'Sofía V', 'Hernán P', 'María R'])]

In [63]:
df_prod1["Año"] = df_prod1["Periodo"].astype(str).str[:4]
df_prod1["Mes"] = df_prod1["Periodo"].astype(str).str[4:6]

In [66]:
df_prod2 = df_prod1[df_prod1["GOS"].isin(["Sebastián S"])]

In [69]:
df_graf1 = df_prod2.groupby(["Año", "Mes"]).agg({"VtaNeta": "sum", "JEq": "sum", "VtaNetaMeta": "sum", "JEqMeta":"sum"}).reset_index()
df_graf1["Prod"] = df_graf1["VtaNeta"] / df_graf1["JEq"]
df_graf1["ProdMeta"] = df_graf1["VtaNetaMeta"] / df_graf1["JEqMeta"]
df_graf1["CumpProd"] = (df_graf1["Prod"] /df_graf1["ProdMeta"])-1
df_graf2 = df_graf1[df_graf1["VtaNeta"]>0]

In [75]:
df_graf2

,Año,Mes,VtaNeta,JEq,VtaNetaMeta,JEqMeta,Prod,ProdMeta,CumpProd
0,2025,01,235961254.0,2744.331324,242459367.0,2723.928091,85981.328837,89010.927944,-0.034036
1,2025,02,225853148.0,2819.268073,234798286.0,2779.525941,80110.561379,84474.220067,-0.051657
2,2025,03,285126205.0,2858.376660,293347143.0,2896.992309,99751.096136,101259.206687,-0.014894
3,2025,04,233928561.0,2725.345292,255061487.0,2842.764867,85834.467183,89723.033369,-0.043340
4,2025,05,245101827.0,2766.790410,251749063.0,2903.129622,88587.059619,86716.439071,0.021572
5,2025,06,223085444.0,2689.712278,251935022.0,2484.180348,82940.263107,101415.753579,-0.182176
6,2025,07,240654475.0,2719.583125,268325461.0,2593.850500,88489.472077,103446.771879,-0.144589
7,2025,08,233776018.0,2572.168017,249254895.0,2509.172680,90886.760278,99337.481635,-0.085071
8,2025,09,213242049.0,2555.872521,242625782.0,2508.163627,83432.192827,96734.431255,-0.137513
9,2025,10,218652517.0,2554.856989,250876056.0,2516.696961,85583.074873,99684.650115,-0.141462


In [131]:
import pandas as pd
import plotly.graph_objects as go

meses = {
    "01":"ENE","02":"FEB","03":"MAR","04":"ABR",
    "05":"MAY","06":"JUN","07":"JUL","08":"AGO",
    "09":"SEP","10":"OCT","11":"NOV","12":"DIC"
}

df = df_graf2.copy()

df["Año"] = df["Año"].astype(str)
df["Mes"] = df["Mes"].astype(str).str.zfill(2)

prod = df.pivot(index="Mes", columns="Año", values="Prod")
cump = df.pivot(index="Mes", columns="Año", values="CumpProd")

orden = [f"{i:02d}" for i in range(1,13)]

prod = prod.reindex(orden)
cump = cump.reindex(orden)

x = [meses[m] for m in orden]

offset = 0.20
cump_plot = cump["2026"] + offset

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x,
        y=prod["2025"],
        mode="lines+markers",
        name="Prod 2025",
        line=dict(color="#efa686", width=4),
        marker=dict(
            size=10,
            color="white",
            line=dict(color="#efa686", width=3)
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=x,
        y=prod["2026"],
        mode="lines+markers",
        name="Prod 2026",
        line=dict(color="#006077", width=4),
        marker=dict(
            size=10,
            color="white",
            line=dict(color="#006077", width=3)
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=x,
        y=cump_plot,
        mode="lines+markers",
        name="% Cump. Meta",
        yaxis="y2",
        line=dict(color="#808080", width=3, dash="dot"),
        marker=dict(
            size=10,
            color="white",
            line=dict(color="#808080", width=3)
        ),
        hovertemplate="%{customdata:.1%}<extra></extra>",
        customdata=cump["2026"]
    )
)

def formato_k(v):
    if pd.isna(v):
        return ""
    return f"{v/1000:.0f}K"

annotations = []

for xi, yi, v in zip(x, cump_plot, cump["2026"]):

    if pd.isna(v):
        continue

    annotations.append(
        dict(
            x=xi,
            y=yi,
            yref="y2",
            text=f"<b>{v:.1%}</b>",
            showarrow=False,
            yshift=18,
            bgcolor="#06b57c" if v >= 0 else "#f43959",
            bordercolor="#06b57c" if v >= 0 else "#f43959",
            borderpad=4,
            font=dict(
                color="white",
                size=10
            )
        )
    )

for xi, yi in zip(x, prod["2026"]):

    if pd.isna(yi):
        continue

    annotations.append(
        dict(
            x=xi,
            y=yi,
            yref="y",
            text=f"<b>{formato_k(yi)}</b>",
            showarrow=False,
            yshift=22,
            bgcolor="#006077",
            bordercolor="#006077",
            borderpad=4,
            font=dict(
                color="white",
                size=12
            )
        )
    )

fig.update_layout(

    annotations=annotations,

    template="plotly_white",

    title=dict(
        text="<b>Productividad | Sebastián S</b>",
        x=0.5,
        font=dict(size=24)
    ),

    height=450,
    width=1226,
    hovermode="x unified",

    legend=dict(
        orientation="v",
        x=1.02,
        y=0.98,
        xanchor="left",
        yanchor="top",
        font=dict(size=13)
    ),

    margin=dict(
        l=60,
        r=60,
        t=120,
        b=50
    ),

    xaxis=dict(
        tickfont=dict(size=13)
    ),

    yaxis=dict(
        title="Productividad",
        showgrid=True,
        gridcolor="#ECECEC",
        zeroline=False,
        tickformat=".0s"
    ),

    yaxis2=dict(
        overlaying="y",
        side="right",
        range=[offset-0.8, offset+0.1],
        showgrid=False,
        showticklabels=False,
        ticks="",
        zeroline=False
    )
)

fig.show()

fig.write_html(
    "Productividad_SebastianS.html",
    include_plotlyjs="cdn",
    full_html=True,
    config={
        "displayModeBar": False,
        "responsive": True
    }
)

In [115]:
df_graf3 = df_prod1[(df_prod1["Año"]=="2026")&(df_prod1["Periodo"]!=202607)].groupby(["GOS", "GOR", "Periodo"]).agg({"VtaNeta": "sum", "JEq": "sum", "VtaNetaMeta": "sum", "JEqMeta":"sum"}).reset_index()

In [129]:
df_graf3["Prod"] = df_graf3["VtaNeta"] / df_graf3["JEq"]
df_graf3["ProdMeta"] = df_graf3["VtaNetaMeta"] / df_graf3["JEqMeta"]
df_graf3["CumpProd"] = (df_graf3["Prod"] /df_graf3["ProdMeta"])-1


In [198]:
dfff = pd.read_excel(r"datos\velocidad3.xlsx")

In [200]:
df_graf3

,GOS,GOR,Tienda,VtaNeta,JEq,VtaNetaMeta,JEqMeta
0,Fátima L,Fátima L,P138 Jiron de la Union - PVE,131548.0,5.362097,131277.0,6.442739
1,Fátima L,Fátima L,P180 Centro Trujillo - PVE,661304.0,11.471007,651821.0,10.653753
2,Fátima L,Fátima L,P181 Supermercados SKA - PVE,647603.0,9.890368,689063.0,10.042294
3,Fátima L,Fátima L,P191 Cine Rimac - PVS,673473.0,7.912625,594707.0,10.042294
4,Fátima L,Fátima L,P193 Trujillo Mansiche - PVS,909515.0,10.527708,1000315.0,11.712938
...,...,...,...,...,...,...,...
108,Sofía V,Melany A,P192 Sullana - PVH,2564085.0,36.373701,2748518.0,38.914030
109,Sofía V,Melany A,P226 Paita - PVH,4724789.0,53.859264,4858034.0,48.493991
110,Sofía V,Melany A,P262 Talara Municipalidad - PVH,4853515.0,53.879000,4387375.0,53.343390
111,Sofía V,Melany A,P724 Tumbes - PVH,5474442.0,58.003313,6405068.0,63.042189


In [208]:
dfff["CumpVelocidad"] = (dfff["Unid x Minuto"] / dfff["Meta Promedio"])-1

In [217]:
dfff["GOR"].unique()
dic_gor = {
    "Fátima León": "Fátima L",
    "DANIEL RODRIGUEZ": "Daniel R",
    "María Rojas": "María R",
    "JOHANNA VILCA": "Johanna V",
    "RODOLFO OLIVRY": "Rodolfo O",
    "VIK ENCISO": "Vik E",
    "JOE HUAMANCONDOR": "Joe H",
    "MELANY ALVARADO": "Melany A",
    "PEPE ARAMBURU": "Pepe A"
}

In [218]:
dfff["GOR"] = dfff["supervisor"]
dfff["GOS"] = dfff["regional"]
dfff["Mes"] = dfff["Mes"].str[0:3]
dfff1 = dfff.dropna()
dic_gos = {
    'Fátima León': "Fátima L",
    'Sebastián Santangelo': "Sebastián S",
    'Sofía Villanueva': "Sofía V",
    'Hernán Perroni': "Hernán P",
    'María Rojas': "María R"
}

dfff1["GOS"] = dfff1["GOS"].map(dic_gos)
dfff1["GOR"] = dfff1["GOR"].map(dic_gor)
dfff1

,Año,Mes,Meta Promedio,Unid x Minuto,regional,supervisor,GOR,GOS,CumpVelocidad
0,2026,ene,13.86,12.14,Fátima León,Fátima León,Fátima L,Fátima L,-0.124098
1,2026,ene,13.13,11.38,Hernán Perroni,DANIEL RODRIGUEZ,Daniel R,Hernán P,-0.133283
2,2026,ene,9.14,8.92,María Rojas,María Rojas,María R,María R,-0.024070
3,2026,ene,13.13,10.04,Sebastián Santangelo,JOHANNA VILCA,Johanna V,Sebastián S,-0.235339
4,2026,ene,14.07,11.27,Sebastián Santangelo,RODOLFO OLIVRY,Rodolfo O,Sebastián S,-0.199005
5,2026,ene,14.17,11.86,Sebastián Santangelo,VIK ENCISO,Vik E,Sebastián S,-0.163020
6,2026,ene,14.73,12.91,Sofía Villanueva,JOE HUAMANCONDOR,Joe H,Sofía V,-0.123557
7,2026,ene,15.08,13.58,Sofía Villanueva,MELANY ALVARADO,Melany A,Sofía V,-0.099469
8,2026,ene,14.91,12.87,Sofía Villanueva,PEPE ARAMBURU,Pepe A,Sofía V,-0.136821
9,2026,feb,13.86,12.71,Fátima León,Fátima León,Fátima L,Fátima L,-0.082973


In [219]:
import plotly.graph_objects as go
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex

tabla = dfff1.pivot_table(
    index=["GOS","GOR"],
    columns="Mes",
    values="CumpVelocidad",
    aggfunc="first"
).reset_index()

orden_gos = [
    "Sebastián S",
    "Sofía V",
    "Hernán P",
    "María R",
    "Fátima L"
]

tabla["GOS"] = pd.Categorical(
    tabla["GOS"],
    categories=orden_gos,
    ordered=True
)

tabla = tabla.sort_values("GOS").reset_index(drop=True)

orden = ["GOS","GOR","ene","feb","mar","abr","may","jun"]
tabla = tabla.reindex(columns=[c for c in orden if c in tabla.columns])

cmap = LinearSegmentedColormap.from_list(
    "",
    ["#e60000","#fdeaea","#ffffff","#d9f2d9","#3cb44b"]
)

norm = Normalize(vmin=-0.15, vmax=0.15)

fill_colors = []

for col in tabla.columns:

    colores = []

    if col in ["GOS","GOR"]:
        colores = ["#ffffff"] * len(tabla)

    else:

        for v in tabla[col]:

            if pd.isna(v):
                colores.append("#ffffff")
            else:
                colores.append(to_hex(cmap(norm(v))))

    fill_colors.append(colores)

valores = []

for col in tabla.columns:

    if col in ["GOS","GOR"]:
        valores.append(tabla[col])

    else:

        txt = []

        for v in tabla[col]:

            if pd.isna(v):
                txt.append("")
            else:

                flecha = "▲" if v >= 0 else "▼"

                txt.append(f"<b>{v:.1%} {flecha}</b>")

        valores.append(txt)

fig = go.Figure(

    data=[
        go.Table(

            columnwidth=[130,110,75,75,75,75,75,75],

            header=dict(

                values=[f"<b>{c}</b>" for c in tabla.columns],
                fill_color="#173A70",
                font=dict(color="white",size=16),
                align="center",
                height=40

            ),

            cells=dict(
                values=valores,
                fill_color=fill_colors,
                align="center",
                height=35,
                font=dict(
                    size=14,
                    color="black"
                )

            )

        )
    ]

)

fig.update_layout(
    margin=dict(l=5,r=5,t=5,b=5),
    height=430
)

fig.show()

In [125]:
import pandas as pd
import plotly.graph_objects as go
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex

meses = {
    "202601":"Ene",
    "202602":"Feb",
    "202603":"Mar",
    "202604":"Abr",
    "202605":"May",
    "202606":"Jun",
    "202607":"Jul",
    "202608":"Ago",
    "202609":"Sep",
    "202610":"Oct",
    "202611":"Nov",
    "202612":"Dic"
}

df = df_graf3.copy()

df["Periodo"] = df["Periodo"].astype(str)
df["Mes"] = df["Periodo"].map(meses)




tabla = df.pivot_table(
    index=["GOS","GOR"],
    columns="Mes",
    values="CumpProd",
    aggfunc="first"
).reset_index()

orden_gos = [
    "Sebastián S",
    "Sofía V",
    "Hernán P",
    "María R",
    "Fátima L"
]

tabla["GOS"] = pd.Categorical(
    tabla["GOS"],
    categories=orden_gos,
    ordered=True
)

tabla = tabla.sort_values("GOS").reset_index(drop=True)

orden = ["GOS","GOR","Ene","Feb","Mar","Abr","May","Jun"]
tabla = tabla.reindex(columns=[c for c in orden if c in tabla.columns])

cmap = LinearSegmentedColormap.from_list(
    "",
    ["#e60000","#fdeaea","#ffffff","#d9f2d9","#3cb44b"]
)

norm = Normalize(vmin=-0.15, vmax=0.15)

fill_colors = []

for col in tabla.columns:

    colores = []

    if col in ["GOS","GOR"]:
        colores = ["#ffffff"] * len(tabla)

    else:

        for v in tabla[col]:

            if pd.isna(v):
                colores.append("#ffffff")
            else:
                colores.append(to_hex(cmap(norm(v))))

    fill_colors.append(colores)

valores = []

for col in tabla.columns:

    if col in ["GOS","GOR"]:
        valores.append(tabla[col])

    else:

        txt = []

        for v in tabla[col]:

            if pd.isna(v):
                txt.append("")
            else:

                flecha = "▲" if v >= 0 else "▼"

                txt.append(f"<b>{v:.1%} {flecha}</b>")

        valores.append(txt)

fig = go.Figure(

    data=[
        go.Table(

            columnwidth=[130,110,75,75,75,75,75,75],

            header=dict(

                values=[f"<b>{c}</b>" for c in tabla.columns],
                fill_color="#173A70",
                font=dict(color="white",size=16),
                align="center",
                height=40

            ),

            cells=dict(
                values=valores,
                fill_color=fill_colors,
                align="center",
                height=35,
                font=dict(
                    size=14,
                    color="black"
                )

            )

        )
    ]

)

fig.update_layout(
    margin=dict(l=5,r=5,t=5,b=5),
    height=430
)

fig.show()

In [221]:
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex
import pandas as pd

tabla = (
    dfff1.pivot_table(
        index=["GOS","GOR"],
        columns="Mes",
        values="CumpVelocidad",
        aggfunc="first"
    )
    .reset_index()
)

orden_gos = [
    "Sebastián S",
    "Sofía V",
    "Hernán P",
    "María R",
    "Fátima L"
]

tabla["GOS"] = pd.Categorical(
    tabla["GOS"],
    categories=orden_gos,
    ordered=True
)

tabla = tabla.sort_values(["GOS","GOR"])

meses_cols = ["ene","feb","mar","abr","may","jun"]
tabla = tabla.reindex(columns=["GOS","GOR"] + meses_cols)

cmap = LinearSegmentedColormap.from_list(
    "",
    [
        "#e60000",
        "#fdeaea",
        "#dff2df",
        "#3cb44b"
    ]
)

norm = Normalize(vmin=-0.15, vmax=0.15)

rowspan = tabla.groupby("GOS").size().to_dict()

html = """
<html>

<head>

<meta charset="utf-8">

<style>

body{
font-family:Segoe UI;
background:white;
margin:8px;
}

table{
border-collapse:collapse;
margin:auto;
font-size:18px;
border-radius:12px;
overflow:hidden;
}

th{
background:#173A70;
color:white;
padding:12px 18px;
text-align:center;
font-weight:700;
font-size:18px;
}

td{
padding:10px 18px;
text-align:center;
border:1px solid white;
font-weight:600;
}

td.gos{
background:white;
font-size:20px;
font-weight:700;
vertical-align:middle;
}

td.gor{
background:white;
font-size:18px;
text-align:left;
padding-left:14px;
}

</style>

</head>

<body>

<table>

<tr>

<th>GOS</th>
<th>GOR</th>
"""

for m in meses_cols:
    html += f"<th>{m.title()}</th>"

html += "</tr>"

impresos = set()

for _, r in tabla.iterrows():

    html += "<tr>"

    if r["GOS"] not in impresos:

        html += f"""
        <td class='gos' rowspan='{rowspan[r["GOS"]]}'>
        {r["GOS"]}
        </td>
        """

        impresos.add(r["GOS"])

    html += f"""
    <td class="gor">
        <span style="
            color:#5B3F8C;
            font-size:15px;
            margin-right:6px;">
            &#128100;
        </span>
        {r["GOR"]}
    </td>
    """

    for m in meses_cols:

        v = r[m]

        if pd.isna(v):

            html += "<td></td>"
            continue

        color = to_hex(cmap(norm(v)))

        flecha = "▲" if v >= 0 else "▼"

        html += f"""
        <td style="
            background:{color};
            font-size:17px;
            font-weight:700;
        ">
        {v:.1%} {flecha}
        </td>
        """

    html += "</tr>"

html += """

</table>

</body>

</html>

"""

with open("heatmap_velocidad.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Archivo generado: heatmap_velocidad.html")

Archivo generado: heatmap_velocidad.html


In [128]:
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex
import pandas as pd

meses = {
    "202601":"Ene",
    "202602":"Feb",
    "202603":"Mar",
    "202604":"Abr",
    "202605":"May",
    "202606":"Jun"
}

df = df_graf3.copy()

df["Periodo"] = df["Periodo"].astype(str)
df["Mes"] = df["Periodo"].map(meses)

tabla = (
    df.pivot_table(
        index=["GOS","GOR"],
        columns="Mes",
        values="CumpProd",
        aggfunc="first"
    )
    .reset_index()
)

orden_gos = [
    "Sebastián S",
    "Sofía V",
    "Hernán P",
    "María R",
    "Fátima L"
]

tabla["GOS"] = pd.Categorical(
    tabla["GOS"],
    categories=orden_gos,
    ordered=True
)

tabla = tabla.sort_values(["GOS","GOR"])

cmap = LinearSegmentedColormap.from_list(
    "",
    ["#e60000","#fdeaea","#ffffff","#d9f2d9","#3cb44b"]
)

norm = Normalize(vmin=-0.15,vmax=0.15)

meses_cols = ["Ene","Feb","Mar","Abr","May","Jun"]

rowspan = tabla.groupby("GOS").size().to_dict()

html = """
<html>

<head>

<style>

body{
font-family:Segoe UI;
background:white;
}

table{
border-collapse:collapse;
margin:auto;
font-size:18px;
border-radius:10px;
overflow:hidden;
}

th{
background:#173A70;
color:white;
padding:12px 18px;
text-align:center;
font-weight:700;
}

td{
padding:10px 18px;
text-align:center;
border:1px solid white;
font-weight:600;
}

td.gos{
background:white;
font-size:20px;
font-weight:700;
vertical-align:middle;
}

td.gor{
background:white;
font-size:18px;
}

</style>

</head>

<body>

<table>

<tr>

<th>GOS</th>
<th>GOR</th>
"""

for m in meses_cols:
    html += f"<th>{m}</th>"

html += "</tr>"

impresos = set()

for _, r in tabla.iterrows():

    html += "<tr>"

    if r["GOS"] not in impresos:

        html += f"""
        <td class='gos' rowspan='{rowspan[r["GOS"]]}'>
        {r["GOS"]}
        </td>
        """

        impresos.add(r["GOS"])

    html += f"""
            <td class='gor' style="text-align:left; padding-left:14px;">
                <span style="
                    color:#5B3F8C;
                    font-size:15px;
                    margin-right:6px;
                ">
                    &#128100;
                </span>
                {r['GOR']}
            </td>
            """

    for m in meses_cols:

        v = r[m]

        if pd.isna(v):
            html += "<td></td>"
            continue

        color = to_hex(cmap(norm(v)))

        texto = "▲" if v>=0 else "▼"

        html += f"""
        <td style='background:{color};'>
        {v:.1%} {texto}
        </td>
        """

    html += "</tr>"

html += """
</table>

</body>

</html>
"""

with open("heatmap_gor.html","w",encoding="utf-8") as f:
    f.write(html)

print("Archivo generado: heatmap_gor.html")

Archivo generado: heatmap_gor.html


In [176]:
df_prod2 = pd.read_csv(
    r"datos\bd_productividad2.csv",
    encoding="latin1",
    sep="\t"
)

In [178]:
df_prod2.columns
df_prod2["VtaNeta"] = df_prod2['VtaNeta']
df_prod2["JEq"] = df_prod2["JEq"]
df_prod2["VtaNetaMeta"] = df_prod2['VtaNetaMeta']
df_prod2["JEqMeta"] = df_prod2['JEqMeta']

print(df_prod2[["VtaNeta", "JEq", "VtaNetaMeta", "JEqMeta"]].dtypes)

VtaNeta          int64
JEq            float64
VtaNetaMeta    float64
JEqMeta        float64
dtype: object


In [182]:
df_prod2.groupby(["GOS", "GOR"]).size() 

GOS          GOR      
Fátima L     Fátima L     14
Hernán P     Daniel R     16
María R      María R       8
Sebastián S  Johanna V     8
             Rodolfo O    15
             Vik E        18
Sofía V      Joe H        11
             José A       11
             Melany A     11
dtype: int64

In [196]:
df_prod1

df_graf4 = df_prod2.groupby(["GOS", "GOR", "Tienda"]).agg({"VtaNeta": "sum", "JEq": "sum", "VtaNetaMeta": "sum", "JEqMeta":"sum"}).reset_index()
df_graf4["Prod"] = df_graf4["VtaNeta"] / df_graf4["JEq"]
df_graf4["ProdMeta"] = df_graf4["VtaNetaMeta"] / df_graf4["JEqMeta"]
df_graf4["CumpProd"] = (df_graf4["Prod"] /df_graf4["ProdMeta"])
df_graf4["CumpVenta"] = (df_graf4["VtaNeta"] /df_graf4["VtaNetaMeta"])
df_graf4["CumpJeq"] = (df_graf4["JEq"] /df_graf4["JEqMeta"])

GOR = "Vik E"
dff_tabla = df_graf4[(df_graf4["GOR"] == GOR)].sort_values("CumpProd", ascending=False).reset_index(drop=True)
dff_tabla["Tienda"] = (
    dff_tabla["Tienda"]
    .str.replace(r"^P\d+\s+", "", regex=True)
    .str.replace(r"\s*-\s*\w+$", "", regex=True)
)

dff_tabla = dff_tabla[(dff_tabla["Tienda"]!="Primavera")]
dff_tabla

,GOS,GOR,Tienda,VtaNeta,JEq,VtaNetaMeta,JEqMeta,Prod,ProdMeta,CumpProd,CumpVenta,CumpJeq
0,Sebastián S,Vik E,Risso,4140198,43.561486,4424901.0,53.715920,95042.625257,82375.969133,1.153766,0.935659,0.810960
1,Sebastián S,Vik E,Centro Civico,5159843,60.777597,5126148.0,68.682661,84897.120584,74635.255686,1.137494,1.006573,0.884905
2,Sebastián S,Vik E,Pro,8495986,73.437340,8274129.0,78.317261,115690.273744,105648.855310,1.095045,1.026813,0.937690
3,Sebastián S,Vik E,1199 San Miguel,5577223,67.303639,5670451.0,72.494657,82866.589266,78218.881930,1.059419,0.983559,0.928394
4,Sebastián S,Vik E,Puente Piedra,10129694,87.874597,9504324.0,86.393285,115274.428794,110012.300311,1.047832,1.065798,1.017146
5,Sebastián S,Vik E,Rimac Alcazar,5025412,57.418278,4789990.0,56.772924,87522.861958,84371.028712,1.037357,1.049149,1.011367
6,Sebastián S,Vik E,Brasil,3580378,51.006375,3619861.0,52.842491,70194.715857,68502.845833,1.024698,0.989093,0.965253
7,Sebastián S,Vik E,Comas,5717895,66.175278,5388089.0,63.845935,86405.304093,84392.044773,1.023856,1.061210,1.036484
8,Sebastián S,Vik E,Brena,6280927,69.985319,6130649.0,69.114628,89746.350381,88702.626148,1.011767,1.024513,1.012598
9,Sebastián S,Vik E,Acho,1907935,31.078007,1914842.0,31.443466,61391.806871,60897.931047,1.008110,0.996393,0.988377


In [197]:

df = dff_tabla.copy()

df = df.sort_values("CumpProd", ascending=False).reset_index(drop=True)

gor = df.loc[0, "GOR"]

max_val = max(df["CumpProd"].abs().max(), 0.01)

filas = ""

for _, r in df.iterrows():

    if r["CumpProd"] > 1:
        color = "#2E7D32"
    elif r["CumpProd"] >= 0.95:
        color = "#F9A825"
    else:
        color = "#C62828"

    ancho = max(8, (abs(r["CumpProd"]) / max_val) * 100)

    texto = (
        f"{r['CumpProd']*100:.0f}% | "
        f"{r['CumpVenta']*100:.0f}% | "
        f"{r['Prod']/1000:.0f}K"
    )

    filas += f"""
    <div class="fila">
        <div class="tienda">{r['Tienda']}</div>

        <div class="barra-fondo">
            <div class="barra"
                 style="width:{ancho:.1f}%; background:{color};">
                {texto}
            </div>
        </div>
    </div>
    """

html = f"""
<!DOCTYPE html>
<html lang="es">

<head>

<meta charset="UTF-8">

<style>

html,body{{
    margin:0;
    padding:0;
    width:320px;
    height:450px;
    overflow:hidden;
    background:#ffffff;
    font-family:'Segoe UI',Arial,sans-serif;
}}

.contenedor{{
    width:320px;
    height:450px;
    box-sizing:border-box;
    padding:10px;
}}

.titulo{{
    font-size:18px;
    font-weight:700;
    color:#17375E;
    margin-bottom:2px;
}}

.subtitulo{{
    font-size:10px;
    color:#666666;
    margin-bottom:10px;
}}

.fila{{
    display:flex;
    align-items:center;
    margin-bottom:8px;
}}

.tienda{{
    width:82px;
    font-size:12px;
    font-weight:600;
    color:#333333;
    overflow:hidden;
    white-space:nowrap;
    text-overflow:ellipsis;
    padding-right:5px;
}}

.barra-fondo{{
    width:168px;
    height:20px;
    background:#EAEAEA;
    border-radius:3px;
    overflow:hidden;
}}

.barra{{
    height:20px;
    display:flex;
    align-items:center;
    justify-content:center;
    color:white;
    font-size:10px;
    font-weight:700;
    border-radius:3px;
    box-sizing:border-box;
}}

</style>

</head>

<body>

<div class="contenedor">

<div class="titulo">
📊 Ranking GOR = {gor}
</div>

<div class="subtitulo">
CumpProd | CumpVenta | Productividad
</div>

{filas}

</div>

</body>

</html>
"""

with open("ranking_gor3.html", "w", encoding="utf-8") as f:
    f.write(html)

print("HTML generado correctamente.")


HTML generado correctamente.
